# Experiment: claude_code
## 0. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Global random seed
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Directory setup
EVIDENCE_DIR = Path('evidence_claude_code')
EDA_PICS_DIR = EVIDENCE_DIR / 'EDA_claude_code_Pics'
EVIDENCE_DIR.mkdir(exist_ok=True)
EDA_PICS_DIR.mkdir(exist_ok=True)

DATA_DIR = Path('../../data')
print('Setup complete. Random seed:', RANDOM_STATE)

Setup complete. Random seed: 42


In [2]:
# Read the variable dictionary to understand variable meanings
with open(DATA_DIR / 'participation_2024-25_data_dictionary_cleaned.txt', 'r') as f:
    data_dict_text = f.read()
print(data_dict_text)

UK Data Archive Data Dictionary

File-level information:

File Name = 		participation_2024-25_annual_data_open
Number of cases = 	34378


Variable-level information:

Pos. = 1,175	Variable = CARTS_NET	Variable label = In the last 12 months, engaged (attended OR participated) with the arts physically
This variable is    numeric, the SPSS measurement level is NOMINAL
SPSS user missing values = -1.7976931348623155e+308 thru -1.0
	Value label information for CARTS_NET
	Value = -3.0	Label = Not applicable
	Value = 1.0	Label = Yes
	Value = 2.0	Label = No
	Value = 3.0	Label = No & Missing

Pos. = 1,308	Variable = AGEBAND	Variable label = Respondent age band (ALL)
This variable is    numeric, the SPSS measurement level is SCALE
	Value label information for AGEBAND
	Value = -3.0	Label = Not Applicable
	Value = 1.0	Label = 16 to 19
	Value = 2.0	Label = 20 to 24
	Value = 3.0	Label = 25 to 29
	Value = 4.0	Label = 30 to 34
	Value = 5.0	Label = 35 to 39
	Value = 6.0	Label = 40 to 44
	Value = 7.0	Lab

# 1. Dataset Ingestion, Schema Checks, and Problem Definition
## 1.1 Dataset Ingestion and Schema Checks

This subsection loads the dataset from the tab-separated file and performs schema validation checks including:
- Verifying the expected number of variables are present
- Confirming all 11 required variables exist in the dataset
- Checking data types for each variable
- Inspecting value ranges against the data dictionary
- Checking for duplicate rows

In [3]:
# Load the data
participation_raw = pd.read_csv(DATA_DIR / 'participation_2024-25_experiment.tab', sep='\t')
print(f'Dataset shape: {participation_raw.shape[0]} rows x {participation_raw.shape[1]} columns')
print(f'\nColumn names:\n{list(participation_raw.columns)}')

Dataset shape: 34378 rows x 11 columns

Column names:
['CARTS_NET', 'AGEBAND', 'SEX', 'QWORK', 'EDUCAT3', 'FINHARD', 'CINTOFT', 'gor', 'rur11cat', 'CHILDHH', 'COHAB']


In [4]:
# Schema checks
REQUIRED_VARS = ['CARTS_NET', 'AGEBAND', 'SEX', 'QWORK', 'EDUCAT3', 'FINHARD',
                 'CINTOFT', 'gor', 'rur11cat', 'CHILDHH', 'COHAB']

# Check all required variables are present
missing_vars = [v for v in REQUIRED_VARS if v not in participation_raw.columns]
print(f'Required variables present: {len(REQUIRED_VARS) - len(missing_vars)}/{len(REQUIRED_VARS)}')
if missing_vars:
    print(f'MISSING: {missing_vars}')
else:
    print('All required variables found.')

# Subset to required variables only
participation_raw = participation_raw[REQUIRED_VARS]
print(f'\nSubset shape: {participation_raw.shape}')

# Data types
print(f'\nData types:\n{participation_raw.dtypes}')

# Value ranges
print(f'\nValue ranges:')
for col in REQUIRED_VARS:
    unique_vals = sorted(participation_raw[col].unique())
    print(f'  {col}: {unique_vals}')

# Duplicates
n_dupes = participation_raw.duplicated().sum()
print(f'\nDuplicate rows: {n_dupes}')

# Basic statistics
print(f'\nDescriptive statistics:')
participation_raw.describe()

Required variables present: 11/11
All required variables found.

Subset shape: (34378, 11)

Data types:
CARTS_NET    int64
AGEBAND      int64
SEX          int64
QWORK        int64
EDUCAT3      int64
FINHARD      int64
CINTOFT      int64
gor          int64
rur11cat     int64
CHILDHH      int64
COHAB        int64
dtype: object

Value ranges:
  CARTS_NET: [1, 2, 3]
  AGEBAND: [-3, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 997]
  SEX: [-5, -4, 1, 2, 997]
  QWORK: [-5, -4, -3, 1, 2, 3, 4, 5, 6, 8, 9, 10, 997, 999]
  EDUCAT3: [-5, -4, -3, 1, 2, 997, 999]
  FINHARD: [-5, -4, -3, 1, 2, 3, 4, 5, 997]
  CINTOFT: [-5, -4, -3, 1, 2, 3, 4, 5]
  gor: [1, 2, 3, 4, 5, 6, 7, 8, 9]
  rur11cat: [1, 2]
  CHILDHH: [-5, -4, 0, 1, 2, 3, 4]
  COHAB: [-4, -3, 1, 2, 997]

Duplicate rows: 8602

Descriptive statistics:


,CARTS_NET,AGEBAND,SEX,QWORK,EDUCAT3,FINHARD,CINTOFT,gor,rur11cat,CHILDHH,COHAB
count,34378.000000,34378.000000,34378.000000,34378.000000,34378.000000,34378.000000,34378.000000,34378.000000,34378.000000,34378.000000,34378.000000
mean,1.090988,24.868957,18.961778,35.033306,29.521816,42.896794,1.466083,5.147420,1.782826,0.495230,11.754174
std,0.291615,128.518102,130.905684,174.710521,167.631995,196.882396,1.411588,2.611578,0.412328,0.996222,115.011315
min,1.000000,-3.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,1.000000,1.000000,-5.000000,-4.000000
25%,1.000000,5.000000,1.000000,1.000000,1.000000,2.000000,1.000000,3.000000,2.000000,0.000000,-3.000000
50%,1.000000,8.000000,1.000000,2.000000,1.000000,2.000000,2.000000,5.000000,2.000000,0.000000,-3.000000
75%,1.000000,11.000000,2.000000,6.000000,2.000000,3.000000,2.000000,7.000000,2.000000,1.000000,1.000000
max,3.000000,997.000000,997.000000,999.000000,999.000000,997.000000,5.000000,9.000000,2.000000,4.000000,997.000000


## 1.2 Problem Definition

### Prediction Task

This is a **binary classification** task: predict whether a respondent **engaged with the arts physically** in the last 12 months.

We frame this as an **under-engagement identification problem** with social research value. Rather than treating arts participation as a purely individual preference, the task investigates whether non-participation is socially patterned across demographic, socioeconomic, digital, and geographic factors. The purpose is to identify groups that may face structural or contextual barriers to physical arts engagement, and to provide evidence for more inclusive cultural policy and public engagement strategies.

### Variables

- **Target variable**: `CARTS_NET` — In the last 12 months, engaged (attended OR participated) with the arts physically
- **Feature variables**: `AGEBAND, SEX, QWORK, EDUCAT3, FINHARD, CINTOFT, gor, rur11cat, CHILDHH, COHAB`

**Note**: For the target variable, rows with values `-3` (Not applicable) and `3` (No & Missing) will be dropped later as missing values, so that the task becomes a clean binary classification problem.

### Variable Dictionary

| Variable | Label | Type | Valid Values |
|----------|-------|------|-------------|
| CARTS_NET | Engaged with arts physically (last 12 months) | Target | 1=Yes, 2=No (drop -3, 3) |
| AGEBAND | Respondent age band | Feature | 1-15 (age bands 16-85+), 997=Prefer not to say |
| SEX | Respondent gender | Feature | 1=Female, 2=Male, -5/-4/-3=missing, 997=Prefer not to say |
| QWORK | Current working status | Feature | 1-10 (employment categories), -5/-4/-3=missing, 997/999=unknown |
| EDUCAT3 | Highest qualification | Feature | 1=Degree+, 2=Other qualification, -5/-4/-3=missing, 997/999=unknown |
| FINHARD | Financial hardship | Feature | 1-5 (comfort scale), -5/-4/-3=missing, 997=Prefer not to say |
| CINTOFT | Internet usage frequency | Feature | 1-5 (frequency scale), -5/-4/-3=missing |
| gor | Government Office Region | Feature | 1-9 (regions) |
| rur11cat | Rural/Urban area | Feature | 1=Rural, 2=Urban |
| CHILDHH | Children in household | Feature | 0-4+ (count), -6/-5/-4/-3=missing, 997=Prefer not to say |
| COHAB | Living as a couple | Feature | 1=Yes, 2=No, -5/-4/-3=missing, 997=Prefer not to say |

# 2. Exploratory Data Analysis

This section explores the dataset to understand distributions, relationships between features and the target, and potential data quality issues. All visualisations are saved to `evidence_claude_code/EDA_claude_code_Pics/`.

First, we prepare the data by removing rows with non-informative target values (-3 and 3) and creating a binary target variable.

In [5]:
# Remove rows where CARTS_NET is -3 or 3
print(f'Rows before filtering: {len(participation_raw)}')
participation_filtered = participation_raw[~participation_raw['CARTS_NET'].isin([-3, 3])].copy()
print(f'Rows after filtering (removing CARTS_NET = -3 or 3): {len(participation_filtered)}')
print(f'Rows removed: {len(participation_raw) - len(participation_filtered)}')

# Create binary target: 1 = Not engaged (under-engaged), 0 = Engaged
# Original: 1 = Yes (engaged), 2 = No (not engaged)
# Recode: 2 -> 1 (under-engaged, positive class), 1 -> 0 (engaged, negative class)
participation_eda = participation_filtered.drop(columns=['CARTS_NET']).copy()
participation_eda['target'] = (participation_filtered['CARTS_NET'] == 2).astype(int)

print(f'\nTarget distribution:')
print(participation_eda['target'].value_counts())
print(f'\nTarget proportions:')
print(participation_eda['target'].value_counts(normalize=True).round(4))

Rows before filtering: 34378
Rows after filtering (removing CARTS_NET = -3 or 3): 34338
Rows removed: 40

Target distribution:
target
0    31290
1     3048
Name: count, dtype: int64

Target proportions:
target
0    0.9112
1    0.0888
Name: proportion, dtype: float64


### 2.1 Target Variable Distribution

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
counts = participation_eda['target'].value_counts().sort_index()
labels = ['Engaged (0)', 'Under-engaged (1)']
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(labels, counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Target Variable Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# Proportion pie
axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, explode=(0, 0.05))
axes[1].set_title('Target Variable Proportions')

plt.tight_layout()
plt.savefig(EDA_PICS_DIR / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: target_distribution.png')

Saved: target_distribution.png


### 2.2 Feature Distributions

In [7]:
FEATURES = ['AGEBAND', 'SEX', 'QWORK', 'EDUCAT3', 'FINHARD', 'CINTOFT',
            'gor', 'rur11cat', 'CHILDHH', 'COHAB']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    participation_eda[col].value_counts().sort_index().plot(kind='bar', ax=axes[i],
        color='steelblue', edgecolor='black', linewidth=0.3)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)

plt.suptitle('Feature Value Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(EDA_PICS_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: feature_distributions.png')

Saved: feature_distributions.png


### 2.3 Feature-Target Relationships

For each feature, we examine the under-engagement rate across its categories.

In [8]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    grouped = participation_eda.groupby(col)['target'].mean().sort_index()
    grouped.plot(kind='bar', ax=axes[i], color='coral', edgecolor='black', linewidth=0.3)
    axes[i].set_title(f'{col} vs Under-engagement', fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Under-engagement Rate')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].axhline(y=participation_eda['target'].mean(), color='navy',
                     linestyle='--', linewidth=1, alpha=0.7)

plt.suptitle('Under-engagement Rate by Feature Category\n(dashed line = overall rate)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(EDA_PICS_DIR / 'feature_target_relationships.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: feature_target_relationships.png')

Saved: feature_target_relationships.png


### 2.4 Correlation Heatmap

In [9]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = participation_eda[FEATURES + ['target']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, square=True, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap (Features + Target)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_PICS_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: correlation_heatmap.png')

Saved: correlation_heatmap.png


### 2.5 Key EDA Insights

Based on the exploratory analysis:
- **Class imbalance**: The target variable is heavily imbalanced (~91% engaged vs ~9% under-engaged). This will require careful handling during modelling (e.g., stratified splits, class weighting, appropriate metrics).
- **Negative codes**: Several features contain negative codes (-3, -4, -5, -6) representing missing or non-applicable responses. These will need to be addressed in the missingness handling step.
- **High non-response rates**: COHAB (~71% non-answered) and EDUCAT3 (~22% non-answered) have particularly high rates of non-informative codes, requiring a recoding approach rather than simple row deletion.
- **Feature patterns**: Some features show clear variation in under-engagement rates across categories, suggesting predictive value.
- **No strong multicollinearity**: The correlation heatmap does not reveal any problematic multicollinearity among features.

# 3. Missingness Handling

In this dataset, missing values are not stored as `NaN` but are represented by negative codes and special values. Based on the data dictionary, the non-informative codes for each variable are:

| Variable | Non-informative Codes | Meaning |
|----------|----------------------|---------|
| AGEBAND | -3, 997 | Not Applicable, Prefer not to say |
| SEX | -5, -4, -3, 997 | Multi-selected, Not answered, Not Answered, Prefer not to say |
| QWORK | -5, -4, -3, 997, 999 | Multi-selected, Not answered, Not Answered, Prefer not to say, Don't know |
| EDUCAT3 | -5, -4, -3, 997, 999 | Multi-selected, Not answered, Not Answered, Prefer not to say, Don't know |
| FINHARD | -5, -4, -3, 997 | Multi-selected, Not answered, Not Answered, Prefer not to say |
| CINTOFT | -5, -4, -3 | Multi-selected, Not answered, Not Answered |
| gor | -- | No missing codes defined |
| rur11cat | -- | No missing codes defined |
| CHILDHH | -6, -5, -4, -3, 997 | Out of range, Multi-selected, Not answered, Not Answered, Prefer not to say |
| COHAB | -5, -4, -3, 997 | Multi-selected, Not answered, Not Answered, Prefer not to say |

### Tiered Handling Strategy

We adopt a tiered approach based on the non-informative rate for each variable:

- **Tier 1 (rate < 5%)**: Drop rows containing non-informative codes. With a large dataset, this causes minimal data loss.
  - Applies to: AGEBAND, SEX, QWORK, FINHARD, CINTOFT, CHILDHH
- **Tier 2 (rate >= 5%)**: Recode all non-informative codes to a single "Unknown" category (code 0). This preserves data while acknowledging the information gap. The high non-response rate often reflects survey routing (e.g., COHAB was not asked to all respondents) rather than random missingness, so "Unknown" is itself a meaningful category.
  - Applies to: EDUCAT3 (~22%), COHAB (~71%)

This approach balances data preservation with analytical integrity.

In [10]:
# Define non-informative codes per feature
NON_INFORMATIVE = {
    'AGEBAND': [-3, 997],
    'SEX': [-5, -4, -3, 997],
    'QWORK': [-5, -4, -3, 997, 999],
    'EDUCAT3': [-5, -4, -3, 997, 999],
    'FINHARD': [-5, -4, -3, 997],
    'CINTOFT': [-5, -4, -3],
    'gor': [],
    'rur11cat': [],
    'CHILDHH': [-6, -5, -4, -3, 997],
    'COHAB': [-5, -4, -3, 997],
}

# First pass: calculate missingness rates on original data
print('Missingness rates per feature (before handling):')
missingness_records = []
for col, codes in NON_INFORMATIVE.items():
    if not codes:
        n_flagged = 0
        rate = 0.0
    else:
        n_flagged = participation_eda[col].isin(codes).sum()
        rate = n_flagged / len(participation_eda)
    missingness_records.append({
        'feature': col,
        'non_informative_codes': str(codes) if codes else 'None',
        'rows_flagged': n_flagged,
        'rate': round(rate, 4),
        'handling': 'recode_to_unknown' if rate >= 0.05 else ('drop_rows' if codes else 'none')
    })
    print(f'  {col}: {n_flagged} rows ({rate:.2%}) -> {"RECODE" if rate >= 0.05 else "DROP" if codes else "OK"}')

print(f'\nRows before missingness handling: {len(participation_eda)}')

participation_clean = participation_eda.copy()

# Tier 2: Recode high-missingness features to 0 (Unknown)
TIER2_FEATURES = ['EDUCAT3', 'COHAB']
for col in TIER2_FEATURES:
    codes = NON_INFORMATIVE[col]
    mask = participation_clean[col].isin(codes)
    n_recoded = mask.sum()
    participation_clean.loc[mask, col] = 0  # Recode to 0 = Unknown
    print(f'\nTier 2 - {col}: Recoded {n_recoded} rows to 0 (Unknown)')

# Tier 1: Drop rows for low-missingness features
TIER1_FEATURES = ['AGEBAND', 'SEX', 'QWORK', 'FINHARD', 'CINTOFT', 'CHILDHH']
for col in TIER1_FEATURES:
    codes = NON_INFORMATIVE[col]
    if not codes:
        continue
    before = len(participation_clean)
    mask = participation_clean[col].isin(codes)
    participation_clean = participation_clean[~mask]
    dropped = before - len(participation_clean)
    print(f'Tier 1 - {col}: Dropped {dropped} rows')

print(f'\nRows after missingness handling: {len(participation_clean)}')
print(f'Total rows removed: {len(participation_eda) - len(participation_clean)}')
print(f'Data retention rate: {len(participation_clean) / len(participation_eda):.2%}')

# Save missingness summary
missingness_df = pd.DataFrame(missingness_records)
missingness_df.to_csv(EVIDENCE_DIR / 'missingness_handling_summary.csv', index=False)
print(f'\nSaved: missingness_handling_summary.csv')

# Verify no non-informative codes remain
print(f'\nVerification - remaining non-informative codes:')
all_clean = True
for col, codes in NON_INFORMATIVE.items():
    if codes:
        remaining = participation_clean[col].isin(codes).sum()
        if remaining > 0:
            all_clean = False
        print(f'  {col}: {remaining}')
if all_clean:
    print('\nAll non-informative codes handled successfully.')

Missingness rates per feature (before handling):
  AGEBAND: 606 rows (1.76%) -> DROP
  SEX: 638 rows (1.86%) -> DROP
  QWORK: 1364 rows (3.97%) -> DROP
  EDUCAT3: 8239 rows (23.99%) -> RECODE
  FINHARD: 1652 rows (4.81%) -> DROP
  CINTOFT: 2029 rows (5.91%) -> RECODE
  gor: 0 rows (0.00%) -> OK
  rur11cat: 0 rows (0.00%) -> OK
  CHILDHH: 69 rows (0.20%) -> DROP
  COHAB: 24434 rows (71.16%) -> RECODE

Rows before missingness handling: 34338

Tier 2 - EDUCAT3: Recoded 8239 rows to 0 (Unknown)

Tier 2 - COHAB: Recoded 24434 rows to 0 (Unknown)


Tier 1 - AGEBAND: Dropped 606 rows
Tier 1 - SEX: Dropped 361 rows
Tier 1 - QWORK: Dropped 1060 rows
Tier 1 - FINHARD: Dropped 922 rows
Tier 1 - CINTOFT: Dropped 1522 rows
Tier 1 - CHILDHH: Dropped 19 rows

Rows after missingness handling: 29848
Total rows removed: 4490
Data retention rate: 86.92%

Saved: missingness_handling_summary.csv

Verification - remaining non-informative codes:
  AGEBAND: 0
  SEX: 0
  QWORK: 0
  EDUCAT3: 0
  FINHARD: 0
  CINTOFT: 0
  CHILDHH: 0
  COHAB: 0

All non-informative codes handled successfully.


# 4. Modelling

## 4.1 Prepare Modelling Data

We define features and target, create preprocessing pipelines, and split the data into train/validation/test sets with a 70/15/15 ratio using stratified sampling to preserve class proportions.

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Define X and y
X = participation_clean[FEATURES].copy()
y = participation_clean['target'].copy()
print(f'X shape: {X.shape}')
print(f'y distribution:\n{y.value_counts()}')

# All features are numeric-coded categoricals - use OneHotEncoder for all
categorical_features = FEATURES

# Preprocessing for Logistic Regression: OneHotEncoder (+ StandardScaler applied to encoded features)
lr_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error'),
         categorical_features)
    ],
    remainder='drop'
)

# Preprocessing for XGBoost: OneHotEncoder only (no scaling needed)
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error'),
         categorical_features)
    ],
    remainder='drop'
)

# Stratified train/validation/test split: 70/15/15
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15/0.85, random_state=RANDOM_STATE, stratify=y_train_val
)

print(f'\nSplit sizes:')
print(f'  Train: {X_train.shape[0]} ({X_train.shape[0]/len(X):.2%})')
print(f'  Validation: {X_val.shape[0]} ({X_val.shape[0]/len(X):.2%})')
print(f'  Test: {X_test.shape[0]} ({X_test.shape[0]/len(X):.2%})')
print(f'\nTarget distribution in each set:')
for name, ys in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f'  {name}: {ys.value_counts().to_dict()} (positive rate: {ys.mean():.4f})')

X shape: (29848, 10)
y distribution:
target
0    27763
1     2085
Name: count, dtype: int64

Split sizes:
  Train: 20892 (69.99%)
  Validation: 4478 (15.00%)
  Test: 4478 (15.00%)

Target distribution in each set:
  Train: {0: 19433, 1: 1459} (positive rate: 0.0698)
  Val: {0: 4165, 1: 313} (positive rate: 0.0699)
  Test: {0: 4165, 1: 313} (positive rate: 0.0699)


## 4.2 Evaluation Harness

Given the task context - identifying under-engaged populations for policy intervention - the evaluation framework prioritises:

1. **Recall (sensitivity)**: Crucial because failing to identify under-engaged individuals (false negatives) means missing the population the policy aims to reach.
2. **Balanced Accuracy**: Accounts for class imbalance by averaging recall across both classes.
3. **PR-AUC**: More informative than ROC-AUC for imbalanced datasets, focusing on the minority class.
4. **ROC-AUC**: Overall discriminative ability of the model.
5. **Precision**: Important to avoid overwhelming outreach programmes with false positives.
6. **F1 Score**: Harmonic mean of precision and recall, balancing both concerns.

The harness returns a standardised dictionary of metrics and prints a formatted summary, ensuring consistent comparison across all models.

In [12]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, balanced_accuracy_score, roc_auc_score,
                             average_precision_score, confusion_matrix,
                             classification_report)

def evaluate_model(model, X_eval, y_eval, preprocessor, dataset_name='Validation'):
    """Evaluate a model and return a dictionary of metrics."""
    X_transformed = preprocessor.transform(X_eval)
    y_pred = model.predict(X_transformed)
    y_proba = model.predict_proba(X_transformed)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred).ravel()

    metrics = {
        'accuracy': round(accuracy_score(y_eval, y_pred), 4),
        'precision': round(precision_score(y_eval, y_pred, zero_division=0), 4),
        'recall': round(recall_score(y_eval, y_pred), 4),
        'f1': round(f1_score(y_eval, y_pred), 4),
        'balanced_accuracy': round(balanced_accuracy_score(y_eval, y_pred), 4),
        'roc_auc': round(roc_auc_score(y_eval, y_proba), 4),
        'pr_auc': round(average_precision_score(y_eval, y_proba), 4),
        'specificity': round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

    print(f'\n=== {dataset_name} Set Evaluation ===')
    print(f'  Accuracy:          {metrics["accuracy"]:.4f}')
    print(f'  Precision:         {metrics["precision"]:.4f}')
    print(f'  Recall:            {metrics["recall"]:.4f}')
    print(f'  F1 Score:          {metrics["f1"]:.4f}')
    print(f'  Balanced Accuracy: {metrics["balanced_accuracy"]:.4f}')
    print(f'  ROC-AUC:           {metrics["roc_auc"]:.4f}')
    print(f'  PR-AUC:            {metrics["pr_auc"]:.4f}')
    print(f'  Specificity:       {metrics["specificity"]:.4f}')
    print(f'  Confusion Matrix:  TN={tn}, FP={fp}, FN={fn}, TP={tp}')

    return metrics

print('Evaluation harness defined.')

Evaluation harness defined.


## 4.3 Baseline Logistic Regression

We train a baseline Logistic Regression with `class_weight='balanced'` to address the class imbalance, and default regularisation (C=1.0, L2 penalty). Evaluation is performed on the validation set only.

In [13]:
# Fit preprocessor on training data
lr_preprocessor.fit(X_train)

# Baseline LR with balanced class weights
baseline_lr = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='lbfgs',
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)
baseline_lr.fit(lr_preprocessor.transform(X_train), y_train)

# Evaluate on validation set
baseline_lr_metrics = evaluate_model(baseline_lr, X_val, y_val, lr_preprocessor, 'Validation')

# Save metrics
baseline_lr_df = pd.DataFrame([baseline_lr_metrics])
baseline_lr_df.to_csv(EVIDENCE_DIR / 'baseline_lr_validation_metrics.csv', index=False)
print('\nSaved: baseline_lr_validation_metrics.csv')


=== Validation Set Evaluation ===
  Accuracy:          0.7135
  Precision:         0.1541
  Recall:            0.6901
  F1 Score:          0.2519
  Balanced Accuracy: 0.7027
  ROC-AUC:           0.7642
  PR-AUC:            0.2186
  Specificity:       0.7152
  Confusion Matrix:  TN=2979, FP=1186, FN=97, TP=216

Saved: baseline_lr_validation_metrics.csv


# 5. Improving Performance

## 5.1 Tune Logistic Regression

We perform a grid search over regularisation strength (C), penalty type, and class weighting. All evaluation is done on the validation set only.

In [14]:
from itertools import product

# Hyperparameter grid for LR
lr_param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l1', 'l2'],
    'class_weight': ['balanced', None],
}

lr_results = []
best_lr_score = -1
best_lr_model = None
best_lr_params = None

for C, penalty, cw in product(lr_param_grid['C'], lr_param_grid['penalty'],
                                lr_param_grid['class_weight']):
    solver = 'saga'  # supports both l1 and l2
    try:
        model = LogisticRegression(
            C=C, penalty=penalty, solver=solver, class_weight=cw,
            max_iter=2000, random_state=RANDOM_STATE
        )
        model.fit(lr_preprocessor.transform(X_train), y_train)

        X_val_t = lr_preprocessor.transform(X_val)
        y_pred = model.predict(X_val_t)
        y_proba = model.predict_proba(X_val_t)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
        result = {
            'C': C, 'penalty': penalty, 'class_weight': str(cw),
            'accuracy': round(accuracy_score(y_val, y_pred), 4),
            'precision': round(precision_score(y_val, y_pred, zero_division=0), 4),
            'recall': round(recall_score(y_val, y_pred), 4),
            'f1': round(f1_score(y_val, y_pred), 4),
            'balanced_accuracy': round(balanced_accuracy_score(y_val, y_pred), 4),
            'roc_auc': round(roc_auc_score(y_val, y_proba), 4),
            'pr_auc': round(average_precision_score(y_val, y_proba), 4),
        }
        lr_results.append(result)

        # Use balanced_accuracy as the selection criterion
        score = result['balanced_accuracy']
        if score > best_lr_score:
            best_lr_score = score
            best_lr_model = model
            best_lr_params = {'C': C, 'penalty': penalty, 'class_weight': str(cw)}
    except Exception as e:
        pass  # skip invalid combinations

lr_results_df = pd.DataFrame(lr_results)
lr_results_df = lr_results_df.sort_values('balanced_accuracy', ascending=False)
lr_results_df.to_csv(EVIDENCE_DIR / 'lr_tuning_results.csv', index=False)
print(f'Total LR configurations evaluated: {len(lr_results)}')
print(f'Best LR params: {best_lr_params}')
print(f'Best LR balanced accuracy: {best_lr_score:.4f}')
print(f'\nTop 5 configurations:')
lr_results_df.head()

Total LR configurations evaluated: 24
Best LR params: {'C': 1.0, 'penalty': 'l2', 'class_weight': 'balanced'}
Best LR balanced accuracy: 0.7027

Top 5 configurations:


,C,penalty,class_weight,accuracy,precision,recall,f1,balanced_accuracy,roc_auc,pr_auc
14,1.00,l2,balanced,0.7135,0.1541,0.6901,0.2519,0.7027,0.7642,0.2188
10,0.10,l2,balanced,0.7130,0.1534,0.6869,0.2507,0.7010,0.7641,0.2183
4,0.01,l1,balanced,0.6979,0.1482,0.6997,0.2446,0.6987,0.7582,0.2118
6,0.01,l2,balanced,0.7086,0.1507,0.6837,0.2470,0.6971,0.7613,0.2136
16,10.00,l1,balanced,0.7166,0.1531,0.6741,0.2496,0.6970,0.7537,0.2203


## 5.2 Tune XGBoost

We tune XGBoost over key hyperparameters including tree depth, learning rate, number of estimators, and class imbalance handling via `scale_pos_weight`. All evaluation is on the validation set only.

In [15]:
# Calculate scale_pos_weight for class imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
default_spw = neg_count / pos_count
print(f'Default scale_pos_weight: {default_spw:.2f}')

# Fit XGBoost preprocessor
xgb_preprocessor.fit(X_train)

# Hyperparameter grid for XGBoost
xgb_param_grid = {
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'scale_pos_weight': [1, default_spw],
}

xgb_results = []
best_xgb_score = -1
best_xgb_model = None
best_xgb_params = None

for md_, ne, lr_, spw in product(xgb_param_grid['max_depth'],
                                   xgb_param_grid['n_estimators'],
                                   xgb_param_grid['learning_rate'],
                                   xgb_param_grid['scale_pos_weight']):
    model = XGBClassifier(
        max_depth=md_, n_estimators=ne, learning_rate=lr_,
        scale_pos_weight=spw, random_state=RANDOM_STATE,
        eval_metric='logloss', verbosity=0, use_label_encoder=False
    )
    model.fit(xgb_preprocessor.transform(X_train), y_train)

    X_val_t = xgb_preprocessor.transform(X_val)
    y_pred = model.predict(X_val_t)
    y_proba = model.predict_proba(X_val_t)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    result = {
        'max_depth': md_, 'n_estimators': ne, 'learning_rate': lr_,
        'scale_pos_weight': round(spw, 2),
        'accuracy': round(accuracy_score(y_val, y_pred), 4),
        'precision': round(precision_score(y_val, y_pred, zero_division=0), 4),
        'recall': round(recall_score(y_val, y_pred), 4),
        'f1': round(f1_score(y_val, y_pred), 4),
        'balanced_accuracy': round(balanced_accuracy_score(y_val, y_pred), 4),
        'roc_auc': round(roc_auc_score(y_val, y_proba), 4),
        'pr_auc': round(average_precision_score(y_val, y_proba), 4),
    }
    xgb_results.append(result)

    score = result['balanced_accuracy']
    if score > best_xgb_score:
        best_xgb_score = score
        best_xgb_model = model
        best_xgb_params = {'max_depth': md_, 'n_estimators': ne,
                           'learning_rate': lr_, 'scale_pos_weight': round(spw, 2)}

xgb_results_df = pd.DataFrame(xgb_results)
xgb_results_df = xgb_results_df.sort_values('balanced_accuracy', ascending=False)
xgb_results_df.to_csv(EVIDENCE_DIR / 'xgb_tuning_results.csv', index=False)
print(f'Total XGBoost configurations evaluated: {len(xgb_results)}')
print(f'Best XGBoost params: {best_xgb_params}')
print(f'Best XGBoost balanced accuracy: {best_xgb_score:.4f}')
print(f'\nTop 5 configurations:')
xgb_results_df.head()

Default scale_pos_weight: 13.32


Total XGBoost configurations evaluated: 54
Best XGBoost params: {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.01, 'scale_pos_weight': 13.32}
Best XGBoost balanced accuracy: 0.7009

Top 5 configurations:


,max_depth,n_estimators,learning_rate,scale_pos_weight,accuracy,precision,recall,f1,balanced_accuracy,roc_auc,pr_auc
31,5,300,0.01,13.32,0.7157,0.1542,0.6837,0.2516,0.7009,0.7550,0.2026
13,3,300,0.01,13.32,0.7121,0.1529,0.6869,0.2501,0.7005,0.7607,0.2060
7,3,200,0.01,13.32,0.7003,0.1488,0.6965,0.2452,0.6985,0.7582,0.2005
5,3,100,0.20,13.32,0.7213,0.1545,0.6677,0.2509,0.6965,0.7591,0.2099
9,3,200,0.10,13.32,0.7184,0.1535,0.6709,0.2499,0.6964,0.7596,0.2122


## 5.3 Model Comparison on Test Set

Only at this stage do we use the held-out test set. We compare three models:
1. Baseline Logistic Regression
2. Tuned Logistic Regression
3. Tuned XGBoost

In [16]:
print('=== TEST SET COMPARISON ===\n')

test_results = {}

# 1. Baseline LR
print('--- Baseline Logistic Regression ---')
test_results['Baseline LR'] = evaluate_model(baseline_lr, X_test, y_test, lr_preprocessor, 'Test')

# 2. Tuned LR
print('\n--- Tuned Logistic Regression ---')
test_results['Tuned LR'] = evaluate_model(best_lr_model, X_test, y_test, lr_preprocessor, 'Test')

# 3. Tuned XGBoost
print('\n--- Tuned XGBoost ---')
test_results['Tuned XGBoost'] = evaluate_model(best_xgb_model, X_test, y_test, xgb_preprocessor, 'Test')

# Save comparison
test_comparison_df = pd.DataFrame(test_results).T
test_comparison_df.index.name = 'model'
test_comparison_df.to_csv(EVIDENCE_DIR / 'test_model_comparison.csv')
print('\nSaved: test_model_comparison.csv')
print('\nTest set comparison:')
test_comparison_df

=== TEST SET COMPARISON ===

--- Baseline Logistic Regression ---

=== Test Set Evaluation ===
  Accuracy:          0.7086
  Precision:         0.1487
  Recall:            0.6709
  F1 Score:          0.2435
  Balanced Accuracy: 0.6912
  ROC-AUC:           0.7509
  PR-AUC:            0.2036
  Specificity:       0.7114
  Confusion Matrix:  TN=2963, FP=1202, FN=103, TP=210

--- Tuned Logistic Regression ---

=== Test Set Evaluation ===
  Accuracy:          0.7079
  Precision:         0.1484
  Recall:            0.6709
  F1 Score:          0.2431
  Balanced Accuracy: 0.6908
  ROC-AUC:           0.7508
  PR-AUC:            0.2036
  Specificity:       0.7107
  Confusion Matrix:  TN=2960, FP=1205, FN=103, TP=210

--- Tuned XGBoost ---

=== Test Set Evaluation ===
  Accuracy:          0.7052
  Precision:         0.1447
  Recall:            0.6550
  F1 Score:          0.2370
  Balanced Accuracy: 0.6820
  ROC-AUC:           0.7464
  PR-AUC:            0.2011
  Specificity:       0.7090
  Confusi

,accuracy,precision,recall,f1,balanced_accuracy,roc_auc,pr_auc,specificity,tn,fp,fn,tp
model,,,,,,,,,,,,
Baseline LR,0.7086,0.1487,0.6709,0.2435,0.6912,0.7509,0.2036,0.7114,2963.0,1202.0,103.0,210.0
Tuned LR,0.7079,0.1484,0.6709,0.2431,0.6908,0.7508,0.2036,0.7107,2960.0,1205.0,103.0,210.0
Tuned XGBoost,0.7052,0.1447,0.6550,0.2370,0.6820,0.7464,0.2011,0.7090,2953.0,1212.0,108.0,205.0


## 5.4 Final Model Decision

### Model Selection Framework

Given the policy purpose of this task - identifying under-engaged populations - we define a multi-dimensional scoring framework with the following weighted criteria:

| Criterion | Weight | Rationale |
|-----------|--------|-----------|
| Recall | 0.35 | Highest priority: missing under-engaged individuals defeats the purpose |
| Balanced Accuracy | 0.20 | Overall fairness across both classes |
| PR-AUC | 0.15 | Performance on minority class detection |
| ROC-AUC | 0.10 | General discriminative ability |
| F1 Score | 0.10 | Balance of precision and recall |
| Interpretability | 0.10 | Policy audiences need transparent models |

For interpretability, Logistic Regression scores 1.0 (fully interpretable) and XGBoost scores 0.5 (less transparent but still explainable).

In [17]:
# Model selection framework
WEIGHTS = {
    'recall': 0.35,
    'balanced_accuracy': 0.20,
    'pr_auc': 0.15,
    'roc_auc': 0.10,
    'f1': 0.10,
    'interpretability': 0.10,
}

INTERPRETABILITY = {
    'Baseline LR': 1.0,
    'Tuned LR': 1.0,
    'Tuned XGBoost': 0.5,
}

selection_records = []
for model_name, metrics in test_results.items():
    scores = {}
    weighted_total = 0
    for criterion, weight in WEIGHTS.items():
        if criterion == 'interpretability':
            val = INTERPRETABILITY[model_name]
        else:
            val = metrics[criterion]
        scores[f'{criterion}_score'] = round(val, 4)
        weighted_total += val * weight
    scores['model'] = model_name
    scores['weighted_total'] = round(weighted_total, 4)
    selection_records.append(scores)

selection_df = pd.DataFrame(selection_records)
selection_df = selection_df.sort_values('weighted_total', ascending=False)
selection_df['rank'] = range(1, len(selection_df) + 1)
selection_df.to_csv(EVIDENCE_DIR / 'model_selection_framework.csv', index=False)

print('Model Selection Framework Results:')
print(selection_df.to_string(index=False))

final_model_name = selection_df.iloc[0]['model']
print(f'\n*** Final Model Selected: {final_model_name} ***')

Model Selection Framework Results:
 recall_score  balanced_accuracy_score  pr_auc_score  roc_auc_score  f1_score  interpretability_score         model  weighted_total  rank
       0.6709                   0.6912        0.2036         0.7509    0.2435                     1.0   Baseline LR          0.6030     1
       0.6709                   0.6908        0.2036         0.7508    0.2431                     1.0      Tuned LR          0.6029     2
       0.6550                   0.6820        0.2011         0.7464    0.2370                     0.5 Tuned XGBoost          0.5442     3

*** Final Model Selected: Baseline LR ***


### Tuning Summaries

In [18]:
# Tuned LR summary
print('=' * 60)
print('TUNED LOGISTIC REGRESSION - TUNING SUMMARY')
print('=' * 60)
print(f'  Tuning method:           Grid Search')
print(f'  Hyperparameters searched: C, penalty, class_weight')
print(f'  C values:                {lr_param_grid["C"]}')
print(f'  Penalty values:          {lr_param_grid["penalty"]}')
print(f'  Class weight values:     {lr_param_grid["class_weight"]}')
print(f'  Total configurations:    {len(lr_results)}')
print(f'  Best hyperparameters:    {best_lr_params}')
print(f'  Best val balanced acc:   {best_lr_score:.4f}')

print()
print('=' * 60)
print('TUNED XGBOOST - TUNING SUMMARY')
print('=' * 60)
print(f'  Tuning method:           Grid Search')
print(f'  Hyperparameters searched: max_depth, n_estimators, learning_rate, scale_pos_weight')
print(f'  max_depth values:        {xgb_param_grid["max_depth"]}')
print(f'  n_estimators values:     {xgb_param_grid["n_estimators"]}')
print(f'  learning_rate values:    {xgb_param_grid["learning_rate"]}')
print(f'  scale_pos_weight values: [1, {default_spw:.2f}]')
print(f'  Total configurations:    {len(xgb_results)}')
print(f'  Best hyperparameters:    {best_xgb_params}')
print(f'  Best val balanced acc:   {best_xgb_score:.4f}')

TUNED LOGISTIC REGRESSION - TUNING SUMMARY
  Tuning method:           Grid Search
  Hyperparameters searched: C, penalty, class_weight
  C values:                [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
  Penalty values:          ['l1', 'l2']
  Class weight values:     ['balanced', None]
  Total configurations:    24
  Best hyperparameters:    {'C': 1.0, 'penalty': 'l2', 'class_weight': 'balanced'}
  Best val balanced acc:   0.7027

TUNED XGBOOST - TUNING SUMMARY
  Tuning method:           Grid Search
  Hyperparameters searched: max_depth, n_estimators, learning_rate, scale_pos_weight
  max_depth values:        [3, 5, 7]
  n_estimators values:     [100, 200, 300]
  learning_rate values:    [0.01, 0.1, 0.2]
  scale_pos_weight values: [1, 13.32]
  Total configurations:    54
  Best hyperparameters:    {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.01, 'scale_pos_weight': 13.32}
  Best val balanced acc:   0.7009
